In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from statsmodels.nonparametric.smoothers_lowess import lowess

import sim_ranking as sr
import ml_tools as mlt

In [ ]:
nzgmdb_ffp = Path("/Users/claudy/dev/work/data/gm_datasets/nz_gmdb/v4.3/custom/mod_ground_motion_im_table_rotd50_flat.csv")
gnn_result_dir = Path("/Users/claudy/dev/work/data/sim_ranking/results/gnn/current/0414_0947_cv_v4p3NZGMDB_v2p10_6e8s")

In [ ]:
# Injected Parameters
gnn_results_dir = "/Users/claudy/dev/work/data/sim_ranking/results/gnn/current/0414_1947_cv_v4p3NZGMDB_v2p9_6e8s"


**Result Directory:** `{python} str(gnn_results_dir)`

In [ ]:
run_config = sr.ml.RunConfig.from_yaml(gnn_result_dir / "run_config.yaml")

obs_data = sr.data.load_obs_nzgmdb(nzgmdb_ffp)
gnn_results = pd.read_parquet(gnn_result_dir / "val_results.parquet").sort_index()

emp_gm_params = pd.read_parquet(run_config.emp_gm_params_ffp)

cim_results = None
if (gnn_result_dir / "cim_results").exists():
    cim_results = pd.read_parquet(gnn_result_dir / "cim_results" / "val_results.parquet").sort_index()
    assert cim_results.index.equals(gnn_results.index), "CIM results and GNN results do not match in index"

print(f"Number of scenarios {gnn_results.shape[0]}")

In [ ]:
# Compute the correlations
emp_gnn_res_site_pair_corrs, emp_obs_res_site_pair_corrs, emp_cim_res_site_pair_corrs, site_pairs_df = sr.analysis.compute_site_int_obs_correlation_residuals(gnn_results, obs_data, emp_gm_params, cim_results=cim_results)

In [ ]:
# Compute correlation residuals with respect to observed correlations
gnn_corr_residuals = emp_obs_res_site_pair_corrs - emp_gnn_res_site_pair_corrs

cim_corr_residuals = None
if emp_cim_res_site_pair_corrs is not None:
    cim_corr_residuals = emp_obs_res_site_pair_corrs - emp_cim_res_site_pair_corrs

In [ ]:
# Apply the fisher z-transform to the correlations
z_emp_gnn_res_site_pair_corrs = pd.DataFrame(data=sr.analysis.fisher_transform(emp_gnn_res_site_pair_corrs.values), index=emp_gnn_res_site_pair_corrs.index, columns=emp_gnn_res_site_pair_corrs.columns)
z_emp_obs_res_site_pair_corrs = pd.DataFrame(data=sr.analysis.fisher_transform(emp_obs_res_site_pair_corrs.values), index=emp_obs_res_site_pair_corrs.index, columns=emp_obs_res_site_pair_corrs.columns)

z_emp_cim_res_site_pair_corrs = None
if emp_cim_res_site_pair_corrs is not None:
    z_emp_cim_res_site_pair_corrs = pd.DataFrame(data=sr.analysis.fisher_transform(emp_cim_res_site_pair_corrs.values), index=emp_cim_res_site_pair_corrs.index, columns=emp_cim_res_site_pair_corrs.columns)

In [ ]:
# Compute the residual for the normalized correlations
z_gnn_corr_residuals = z_emp_obs_res_site_pair_corrs - z_emp_gnn_res_site_pair_corrs

z_cim_corr_residuals = None
if z_emp_cim_res_site_pair_corrs is not None:
    z_cim_corr_residuals = z_emp_obs_res_site_pair_corrs - z_emp_cim_res_site_pair_corrs

### Number Of Predictions/Records Per Site of Interest

In [ ]:
# Number of predictions per site
n_pred_per_site = gnn_results.site_int.value_counts().sort_values()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.bar(np.arange(len(n_pred_per_site)), n_pred_per_site.values)
ax.set_xlim(0, len(n_pred_per_site))
ax.set_ylabel("Number of predictions/records")

fig.tight_layout()

## Site-Pairs

#### Number of events per site-pair

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.bar(np.arange(site_pairs_df.shape[0]), site_pairs_df["count"].sort_values().values)
ax.set_xlim(0, site_pairs_df.shape[0])
ax.set_ylabel("Number of events per site pair")

fig.tight_layout()


#### Number of site-pairs per site of interest

In [ ]:
site_int_n_pairs =  site_pairs_df["site_int"].value_counts()

fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(np.arange(len(site_int_n_pairs)), site_int_n_pairs.sort_values().values)
ax.set_xlim(0, len(site_int_n_pairs))
ax.set_ylabel("Number of site pairs per site")

fig.tight_layout()

#### Site-to-site distance of site pairs

In [ ]:
dist_matrix = sr.utils.calculate_distance_matrix(obs_data.sites, obs_data.site_df)


In [ ]:
site_int_ind = dist_matrix.index.get_indexer_for(site_pairs_df["site_int"].values)
obs_site_ind = dist_matrix.columns.get_indexer_for(site_pairs_df["obs_site"].values)

site_dist = dist_matrix.values[site_int_ind, obs_site_ind]
site_pairs_df["dist"] = site_dist


In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.hist(site_pairs_df["dist"].values, bins=50)
ax.set_xlim(0, site_pairs_df["dist"].max())
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Number of site pairs")

fig.tight_layout()

## Correlation Analysis

### General Residual Trends 

In [ ]:
# gnn_corr_res_bias = gnn_corr_residuals.mean(axis=0)
# gnn_corr_res_std = gnn_corr_residuals.std(axis=0)

# cim_corr_res_bias, cim_corr_res_std = None, None
# if cim_corr_residuals is not None:
#     cim_corr_res_bias = cim_corr_residuals.mean(axis=0)
#     cim_corr_res_std = cim_corr_residuals.std(axis=0)
    

# fig, ax1, ax2 = sr.plot_utils.get_pSA_bias_residual_fig()

# ax1.plot(sr.constants.PERIODS, gnn_corr_res_bias[sr.constants.PSA_KEYS], label="GNN", c="blue")
# ax2.plot(sr.constants.PERIODS, gnn_corr_res_std[sr.constants.PSA_KEYS], label="GNN", c="blue")

# if cim_corr_res_bias is not None:
#     ax1.plot(sr.constants.PERIODS, cim_corr_res_bias[sr.constants.PSA_KEYS], label="cIM", c="green")
#     ax2.plot(sr.constants.PERIODS, cim_corr_res_std[sr.constants.PSA_KEYS], label="cIM", c="green")

# ax1.legend()

#### Fisher Transform Residuals

In [ ]:
z_gnn_corr_res_bias = z_gnn_corr_residuals.mean(axis=0)
z_gnn_corr_res_std = z_gnn_corr_residuals.std(axis=0)

z_cim_corr_res_bias, z_cim_corr_res_std = None, None
if z_cim_corr_residuals is not None:
    z_cim_corr_res_bias = z_cim_corr_residuals.mean(axis=0)
    z_cim_corr_res_std = z_cim_corr_residuals.std(axis=0)
    

fig, ax1, ax2 = sr.plot_utils.get_pSA_bias_residual_fig()

ax1.plot(sr.constants.PERIODS, z_gnn_corr_res_bias[sr.constants.PSA_KEYS], label="GNN", c="blue")
ax2.plot(sr.constants.PERIODS, z_gnn_corr_res_std[sr.constants.PSA_KEYS], label="GNN", c="blue")

if z_cim_corr_res_bias is not None:
    ax1.plot(sr.constants.PERIODS, z_cim_corr_res_bias[sr.constants.PSA_KEYS], label="cIM", c="green")
    ax2.plot(sr.constants.PERIODS, z_cim_corr_res_std[sr.constants.PSA_KEYS], label="cIM", c="green")

ax1.legend()

### Site-To-Site Distance Trends

In [ ]:
plot_ims = ["pSA_0.01", "pSA_0.1", "pSA_0.5", "pSA_1.0", "pSA_3.0", "pSA_10.0"]

# if run_config.im_set == sr.constants.IMSet.all:
    # plot_ims.append("PGA")
    # plot_ims.append("CAV")

In [ ]:
# emp_gnn_res_site_pair_corrs.to_parquet("/Users/claudy/dev/work/tmp/emp_gnn_res_site_pair_corrs.parquet")
# site_pairs_df.to_parquet("/Users/claudy/dev/work/tmp/site_pairs_df.parquet")

#### IM Scatter

In [ ]:

fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), n_cols=2, n_rows=-1, ind_figsize=(8, 6))

for ix,  (cur_ax, cur_im) in enumerate(zip(axs, plot_ims)):
    # emp_gnn_lowess_result = lowess(emp_gnn_res_site_pair_corrs[cur_im].values, site_pairs_df["dist"].values, frac=0.1)
    gnn_lowess_x, gnn_lowess_y = mlt.utils.compute_lowess_bootstrap(site_pairs_df["dist"].values, emp_gnn_res_site_pair_corrs[cur_im].values, n_bootstrap_runs=100, frac=1/5)

    cur_ax.scatter(site_pairs_df["dist"].values, emp_gnn_res_site_pair_corrs[cur_im].values, label="GNN Predicted", c="blue", s=5)
    cur_ax.plot(gnn_lowess_x, gnn_lowess_y.mean(axis=1), label="GNN Lowess", c="blue", lw=2)


    obs_lowess_x, obs_lowess_y = mlt.utils.compute_lowess_bootstrap(site_pairs_df["dist"].values, emp_obs_res_site_pair_corrs[cur_im].values, n_bootstrap_runs=100, frac=1/5)
    cur_ax.scatter(site_pairs_df["dist"].values, emp_obs_res_site_pair_corrs[cur_im].values, label="Observed", c="red", s=5)
    cur_ax.plot(obs_lowess_x, obs_lowess_y.mean(axis=1), label="Observed Lowess", c="red", lw=2)

    if emp_cim_res_site_pair_corrs is not None:
        cim_lowess_x, cim_lowess_y = mlt.utils.compute_lowess_bootstrap(site_pairs_df["dist"].values, emp_cim_res_site_pair_corrs[cur_im].values, n_bootstrap_runs=100, frac=1/5)
        cur_ax.scatter(site_pairs_df["dist"].values, emp_cim_res_site_pair_corrs[cur_im].values, label="CIM Predicted", c="green", s=5)
        cur_ax.plot(cim_lowess_x, cim_lowess_y.mean(axis=1), label="CIM Lowess", c="green", lw=2)

    cur_ax.set_xlim(0, site_pairs_df["dist"].max())
    cur_ax.set_ylim(0, 1)
    cur_ax.set_xlabel("Distance (km)")
    cur_ax.set_ylabel("Correlation")

    cur_ax.set_title(cur_im)

    if ix == 0:
        cur_ax.legend()    


fig.tight_layout()

In [ ]:
# fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), n_cols=2, n_rows=-1, ind_figsize=(8, 6))

# for ix,  (cur_ax, cur_im) in enumerate(zip(axs, plot_ims)):
#     cur_ax.scatter(site_pairs_df["dist"].values, gnn_corr_residuals[cur_im].values, label="GNN Residuals", c="blue", s=5)
#     if cim_corr_residuals is not None:
#         cur_ax.scatter(site_pairs_df["dist"].values, cim_corr_residuals[cur_im].values, label="CIM Residuals", c="green", s=5)

#     cur_ax.set_xlim(0, site_pairs_df["dist"].max())
#     cur_ax.set_ylim(-1, 1)
#     cur_ax.set_xlabel("Distance (km)")
#     cur_ax.set_ylabel("Correlation residual")

#     cur_ax.set_title(cur_im)

# fig.tight_layout()

In [ ]:
dist_bins = [0, 5, 10, 20, 30]
dist_bin_keys = [f"{dist_bins[i]}_{dist_bins[i+1]}" for i in range(len(dist_bins) - 1)]

In [ ]:
# assert gnn_corr_residuals.index.equals(site_pairs_df.index)
# gnn_corr_residuals["dist"] = site_pairs_df["dist"].values
# gnn_corr_residuals["dist_bin"]  = pd.cut(gnn_corr_residuals["dist"], bins=dist_bins, labels=dist_bin_keys)
# gnn_corr_residual_groups = gnn_corr_residuals.groupby("dist_bin", observed=False)

# gnn_res_dist_corr_bias = gnn_corr_residual_groups[sr.constants.PSA_KEYS].mean()
# gnn_res_dist_corr_std = gnn_corr_residual_groups[sr.constants.PSA_KEYS].std()

In [ ]:
# if cim_corr_residuals is not None:
#     cim_corr_residuals["dist"] = site_pairs_df["dist"].values
#     cim_corr_residuals["dist_bin"]  = pd.cut(cim_corr_residuals["dist"], bins=dist_bins, labels=dist_bin_keys)
#     cim_corr_residual_groups = cim_corr_residuals.groupby("dist_bin", observed=False)
#     assert gnn_corr_residual_groups.groups.keys() == cim_corr_residual_groups.groups.keys(), "CIM and GNN groups do not match"

#     cim_res_dist_corr_bias = cim_corr_residual_groups[sr.constants.PSA_KEYS].mean()
#     cim_res_dist_corr_std = cim_corr_residual_groups[sr.constants.PSA_KEYS].std()

In [ ]:
# colors = list(reversed(cm.viridis(np.linspace(0, 1, len(dist_bin_keys))))) 

# fig, ax1, ax2 = sr.plot_utils.get_pSA_bias_residual_fig(figsize=(16, 6), bias_y_axis_limits=(-0.5, 0.5), std_y_axis_limits=(0, 0.5))

# # Bias
# ax1.plot(sr.constants.PERIODS, gnn_corr_res_bias[sr.constants.PSA_KEYS], label="GNN - Overall", c="blue")
# ax1.plot(sr.constants.PERIODS, cim_corr_res_bias[sr.constants.PSA_KEYS], label="cIM - Overall", c="blue", linestyle="--")
# for i, (cur_key, cur_group) in enumerate(gnn_corr_residual_groups):
# 	ax1.semilogx(sr.constants.PERIODS, gnn_res_dist_corr_bias.loc[cur_key, sr.constants.PSA_KEYS], label=f"{cur_key} - N = {gnn_corr_residual_groups.size()[cur_key]}", c=colors[i])

# 	if cim_corr_res_bias is not None:
# 		ax1.semilogx(sr.constants.PERIODS, cim_res_dist_corr_bias.loc[cur_key, sr.constants.PSA_KEYS], c=colors[i], linestyle="--")

# ax1.legend(title="Distance Bin")

# # Residual Standard Deviation
# ax2.plot(sr.constants.PERIODS, gnn_corr_res_std[sr.constants.PSA_KEYS], label="GNN", c="blue")
# ax2.plot(sr.constants.PERIODS, cim_corr_res_std[sr.constants.PSA_KEYS], label="cIM", c="blue", linestyle="--")
# for i, (cur_key, cur_group) in enumerate(gnn_corr_residual_groups):
# 	ax2.semilogx(sr.constants.PERIODS, gnn_res_dist_corr_std.loc[cur_key, sr.constants.PSA_KEYS], label=f"{cur_key} - N = {gnn_corr_residual_groups.size()[cur_key]}", c=colors[i])

# 	if cim_corr_res_std is not None:
# 		ax2.semilogx(sr.constants.PERIODS, cim_res_dist_corr_std.loc[cur_key, sr.constants.PSA_KEYS], label=f"{cur_key} - N = {cim_corr_residual_groups.size()[cur_key]}", c=colors[i], linestyle="--")


#### Fisher Transform Residuals

In [ ]:
assert z_gnn_corr_residuals.index.equals(site_pairs_df.index)
z_gnn_corr_residuals["dist"] = site_pairs_df["dist"].values
z_gnn_corr_residuals["dist_bin"]  = pd.cut(z_gnn_corr_residuals["dist"], bins=dist_bins, labels=dist_bin_keys)
z_gnn_corr_residual_groups = z_gnn_corr_residuals.groupby("dist_bin", observed=False)

gnn_res_dist_corr_bias = z_gnn_corr_residual_groups[sr.constants.PSA_KEYS].mean()
gnn_res_dist_corr_std = z_gnn_corr_residual_groups[sr.constants.PSA_KEYS].std()

In [ ]:
if z_cim_corr_residuals is not None:
    z_cim_corr_residuals["dist"] = site_pairs_df["dist"].values
    z_cim_corr_residuals["dist_bin"]  = pd.cut(z_cim_corr_residuals["dist"], bins=dist_bins, labels=dist_bin_keys)
    z_cim_corr_residual_groups = z_cim_corr_residuals.groupby("dist_bin", observed=False)
    assert z_gnn_corr_residual_groups.groups.keys() == z_cim_corr_residual_groups.groups.keys(), "CIM and GNN groups do not match"

    cim_res_dist_corr_bias = z_cim_corr_residual_groups[sr.constants.PSA_KEYS].mean()
    cim_res_dist_corr_std = z_cim_corr_residual_groups[sr.constants.PSA_KEYS].std()

In [ ]:
colors = list(reversed(cm.viridis(np.linspace(0, 1, len(dist_bin_keys))))) 

fig, ax1, ax2 = sr.plot_utils.get_pSA_bias_residual_fig(figsize=(16, 6), bias_y_axis_limits=(-0.5, 0.5), std_y_axis_limits=(0, 0.5))

# Bias
ax1.plot(sr.constants.PERIODS, z_gnn_corr_res_bias[sr.constants.PSA_KEYS], label="GNN - Overall", c="blue")
ax1.plot(sr.constants.PERIODS, z_cim_corr_res_bias[sr.constants.PSA_KEYS], label="cIM - Overall", c="blue", linestyle="--")
for i, (cur_key, cur_group) in enumerate(z_gnn_corr_residual_groups):
	ax1.semilogx(sr.constants.PERIODS, gnn_res_dist_corr_bias.loc[cur_key, sr.constants.PSA_KEYS], label=f"{cur_key} - N = {z_gnn_corr_residual_groups.size()[cur_key]}", c=colors[i])

	if z_cim_corr_res_bias is not None:
		ax1.semilogx(sr.constants.PERIODS, cim_res_dist_corr_bias.loc[cur_key, sr.constants.PSA_KEYS], c=colors[i], linestyle="--")

ax1.legend(title="Distance Bin")

# Residual Standard Deviation
ax2.plot(sr.constants.PERIODS, z_gnn_corr_res_std[sr.constants.PSA_KEYS], label="GNN", c="blue")
ax2.plot(sr.constants.PERIODS, z_cim_corr_res_std[sr.constants.PSA_KEYS], label="cIM", c="blue", linestyle="--")
for i, (cur_key, cur_group) in enumerate(z_gnn_corr_residual_groups):
	ax2.semilogx(sr.constants.PERIODS, gnn_res_dist_corr_std.loc[cur_key, sr.constants.PSA_KEYS], label=f"{cur_key} - N = {z_gnn_corr_residual_groups.size()[cur_key]}", c=colors[i])

	if z_cim_corr_res_std is not None:
		ax2.semilogx(sr.constants.PERIODS, cim_res_dist_corr_std.loc[cur_key, sr.constants.PSA_KEYS], label=f"{cur_key} - N = {z_cim_corr_residual_groups.size()[cur_key]}", c=colors[i], linestyle="--")


### $V_{S30}$  Trends

In [ ]:
site_pairs_df["site_int_vs30"] = obs_data.site_df.loc[site_pairs_df["site_int"].values, "vs30"].values
site_pairs_df["obs_site_vs30"] = obs_data.site_df.loc[site_pairs_df["obs_site"].values, "vs30"].values
site_pairs_df["site_int_ln_vs30"] = np.log(obs_data.site_df.loc[site_pairs_df["site_int"].values, "vs30"].values)
site_pairs_df["obs_site_ln_vs30"] = np.log(obs_data.site_df.loc[site_pairs_df["obs_site"].values, "vs30"].values)

site_pairs_df["ln_vs30_diff"] = site_pairs_df["site_int_ln_vs30"] - site_pairs_df["obs_site_ln_vs30"]
site_pairs_df["abs_ln_vs30_diff"] = np.abs(site_pairs_df["ln_vs30_diff"])

#### $V_{S30}$ Log Difference Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.hist(site_pairs_df["abs_ln_vs30_diff"].values, bins=50)
ax.set_xlim(0, site_pairs_df["abs_ln_vs30_diff"].max())
ax.set_xlabel("ln_vs30_diff")
ax.set_ylabel("Number of site pairs")

fig.tight_layout()

#### IM Scatter

In [ ]:
fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), n_cols=2, n_rows=-1, ind_figsize=(8, 6))

for ix,  (cur_ax, cur_im) in enumerate(zip(axs, plot_ims)):
    # Compute lowess for GNN data
    gnn_lowess_x, gnn_lowess_y = mlt.utils.compute_lowess_bootstrap(site_pairs_df["abs_ln_vs30_diff"].values, 
                                                                    emp_gnn_res_site_pair_corrs[cur_im].values, 
                                                                    n_bootstrap_runs=100, 
                                                                    frac=1/5)

    # Plot GNN data
    cur_ax.scatter(site_pairs_df["abs_ln_vs30_diff"].values, 
                   emp_gnn_res_site_pair_corrs[cur_im].values, 
                   label="GNN Predicted", 
                   c="blue", 
                   s=5)
    cur_ax.plot(gnn_lowess_x, gnn_lowess_y.mean(axis=1), label="GNN Lowess", c="blue", lw=2)

    # Compute lowess for observed data
    obs_lowess_x, obs_lowess_y = mlt.utils.compute_lowess_bootstrap(site_pairs_df["abs_ln_vs30_diff"].values, 
                                                                    emp_obs_res_site_pair_corrs[cur_im].values, 
                                                                    n_bootstrap_runs=100, 
                                                                    frac=1/5)
    
    # Plot observed data
    cur_ax.scatter(site_pairs_df["abs_ln_vs30_diff"].values, 
                   emp_obs_res_site_pair_corrs[cur_im].values, 
                   label="Observed", 
                   c="red", 
                   s=5)
    cur_ax.plot(obs_lowess_x, obs_lowess_y.mean(axis=1), label="Observed Lowess", c="red", lw=2)

    # Plot CIM data if available
    if emp_cim_res_site_pair_corrs is not None:
        cim_lowess_x, cim_lowess_y = mlt.utils.compute_lowess_bootstrap(site_pairs_df["abs_ln_vs30_diff"].values, 
                                                                        emp_cim_res_site_pair_corrs[cur_im].values, 
                                                                        n_bootstrap_runs=100, 
                                                                        frac=1/5)
        cur_ax.scatter(site_pairs_df["abs_ln_vs30_diff"].values, 
                       emp_cim_res_site_pair_corrs[cur_im].values, 
                       label="CIM Predicted", 
                       c="green", 
                       s=5)
        cur_ax.plot(cim_lowess_x, cim_lowess_y.mean(axis=1), label="CIM Lowess", c="green", lw=2)

    # Set limits and labels
    cur_ax.set_xlim(0, site_pairs_df["abs_ln_vs30_diff"].max())
    cur_ax.set_ylim(0, 1)
    cur_ax.set_xlabel("Absolute ln(Vs30) Difference")
    cur_ax.set_ylabel("Correlation")

    cur_ax.set_title(cur_im)

    if ix == 0:
        cur_ax.legend()    

fig.tight_layout()

In [ ]:
# ln_vs30_diff_bins = [0, 0.25, 0.75, 1.5, 2.5]
# ln_vs30_diff_bin_keys = [f"{ln_vs30_diff_bins[i]}_{ln_vs30_diff_bins[i+1]}" for i in range(len(ln_vs30_diff_bins) - 1)]
# assert gnn_corr_residuals.index.equals(site_pairs_df.index)

# gnn_corr_residuals["abs_ln_vs30_diff"] = site_pairs_df["abs_ln_vs30_diff"].values
# gnn_corr_residuals["ln_vs30_diff_bin"]  = pd.cut(gnn_corr_residuals["abs_ln_vs30_diff"], bins=ln_vs30_diff_bins, labels=ln_vs30_diff_bin_keys)
# gnn_corr_residual_groups = gnn_corr_residuals.groupby("ln_vs30_diff_bin", observed=False)

# gnn_res_lnVs30Diff_corr_bias = gnn_corr_residual_groups[sr.constants.PSA_KEYS].mean()
# gnn_res_lnVs30Diff_corr_std = gnn_corr_residual_groups[sr.constants.PSA_KEYS].std()


In [ ]:
# if cim_corr_residuals is not None:
#     cim_corr_residuals["abs_ln_vs30_diff"] = site_pairs_df["abs_ln_vs30_diff"].values
#     cim_corr_residuals["ln_vs30_diff_bin"]  = pd.cut(cim_corr_residuals["abs_ln_vs30_diff"], bins=ln_vs30_diff_bins, labels=ln_vs30_diff_bin_keys)
#     cim_corr_residual_groups = cim_corr_residuals.groupby("ln_vs30_diff_bin", observed=False)
#     assert gnn_corr_residual_groups.groups.keys() == cim_corr_residual_groups.groups.keys(), "CIM and GNN groups do not match"

#     cim_res_lnVs30Diff_corr_bias = cim_corr_residual_groups[sr.constants.PSA_KEYS].mean()
#     cim_res_lnVs30Diff_corr_std = cim_corr_residual_groups[sr.constants.PSA_KEYS].std()

In [ ]:
# colors = list(reversed(cm.viridis(np.linspace(0, 1, len(ln_vs30_diff_bin_keys)))))

# fig, ax1, ax2 = sr.plot_utils.get_pSA_bias_residual_fig(figsize=(16, 6), bias_y_axis_limits=(-0.5, 0.5), std_y_axis_limits=(0, 0.5))

# # Bias
# ax1.plot(sr.constants.PERIODS, gnn_corr_res_bias[sr.constants.PSA_KEYS], label="GNN - Overall", c="blue")
# ax1.plot(sr.constants.PERIODS, cim_corr_res_bias[sr.constants.PSA_KEYS], label="cIM - Overall", c="blue", linestyle="--")
# for i, (cur_key, cur_group) in enumerate(gnn_corr_residual_groups):
#     ax1.semilogx(sr.constants.PERIODS, gnn_res_lnVs30Diff_corr_bias.loc[cur_key, sr.constants.PSA_KEYS], label=f"{cur_key} - N = {gnn_corr_residual_groups.size()[cur_key]}", c=colors[i])

#     if cim_corr_res_bias is not None:
#         ax1.semilogx(sr.constants.PERIODS, cim_res_lnVs30Diff_corr_bias.loc[cur_key, sr.constants.PSA_KEYS], c=colors[i], linestyle="--")

# # ax1.legend(title="ln(Vs30) Diff Bin")

# # Residual Standard Deviation
# ax2.plot(sr.constants.PERIODS, gnn_corr_res_std[sr.constants.PSA_KEYS], label="Std", c="blue")
# for i, (cur_key, cur_group) in enumerate(gnn_corr_residual_groups):
#     ax2.semilogx(sr.constants.PERIODS, gnn_res_lnVs30Diff_corr_std.loc[cur_key, sr.constants.PSA_KEYS], c=colors[i])

#     if cim_corr_res_std is not None:
#         ax2.semilogx(sr.constants.PERIODS, cim_res_lnVs30Diff_corr_std.loc[cur_key, sr.constants.PSA_KEYS], c=colors[i], linestyle="--")

#### Fisher Transform Residuals

In [ ]:
ln_vs30_diff_bins = [0, 0.25, 0.75, 1.5, 2.5]
ln_vs30_diff_bin_keys = [f"{ln_vs30_diff_bins[i]}_{ln_vs30_diff_bins[i+1]}" for i in range(len(ln_vs30_diff_bins) - 1)]
assert z_gnn_corr_residuals.index.equals(site_pairs_df.index)

z_gnn_corr_residuals["abs_ln_vs30_diff"] = site_pairs_df["abs_ln_vs30_diff"].values
z_gnn_corr_residuals["ln_vs30_diff_bin"]  = pd.cut(z_gnn_corr_residuals["abs_ln_vs30_diff"], bins=ln_vs30_diff_bins, labels=ln_vs30_diff_bin_keys)
z_gnn_corr_residual_groups = z_gnn_corr_residuals.groupby("ln_vs30_diff_bin", observed=False)

z_gnn_res_lnVs30Diff_corr_bias = z_gnn_corr_residual_groups[sr.constants.PSA_KEYS].mean()
z_gnn_res_lnVs30Diff_corr_std = z_gnn_corr_residual_groups[sr.constants.PSA_KEYS].std()


In [ ]:
if z_cim_corr_residuals is not None:
    z_cim_corr_residuals["abs_ln_vs30_diff"] = site_pairs_df["abs_ln_vs30_diff"].values
    z_cim_corr_residuals["ln_vs30_diff_bin"]  = pd.cut(z_cim_corr_residuals["abs_ln_vs30_diff"], bins=ln_vs30_diff_bins, labels=ln_vs30_diff_bin_keys)
    z_cim_corr_residual_groups = z_cim_corr_residuals.groupby("ln_vs30_diff_bin", observed=False)
    assert z_gnn_corr_residual_groups.groups.keys() == z_cim_corr_residual_groups.groups.keys(), "CIM and GNN groups do not match"

    z_cim_res_lnVs30Diff_corr_bias = z_cim_corr_residual_groups[sr.constants.PSA_KEYS].mean()
    z_cim_res_lnVs30Diff_corr_std = z_cim_corr_residual_groups[sr.constants.PSA_KEYS].std()

In [ ]:
colors = list(reversed(cm.viridis(np.linspace(0, 1, len(ln_vs30_diff_bin_keys)))))

fig, ax1, ax2 = sr.plot_utils.get_pSA_bias_residual_fig(figsize=(16, 6), bias_y_axis_limits=(-0.5, 0.5), std_y_axis_limits=(0, 0.5))

# Bias
ax1.plot(sr.constants.PERIODS, z_gnn_corr_res_bias[sr.constants.PSA_KEYS], label="GNN - Overall", c="blue")
ax1.plot(sr.constants.PERIODS, z_cim_corr_res_bias[sr.constants.PSA_KEYS], label="cIM - Overall", c="blue", linestyle="--")
for i, (cur_key, cur_group) in enumerate(z_gnn_corr_residual_groups):
    ax1.semilogx(sr.constants.PERIODS, z_gnn_res_lnVs30Diff_corr_bias.loc[cur_key, sr.constants.PSA_KEYS], label=f"{cur_key} - N = {z_gnn_corr_residual_groups.size()[cur_key]}", c=colors[i])

    if z_cim_corr_res_bias is not None:
        ax1.semilogx(sr.constants.PERIODS, z_cim_res_lnVs30Diff_corr_bias.loc[cur_key, sr.constants.PSA_KEYS], c=colors[i], linestyle="--")

ax1.legend()

# Residual Standard Deviation
ax2.plot(sr.constants.PERIODS, z_gnn_corr_res_std[sr.constants.PSA_KEYS], label="Std", c="blue")
for i, (cur_key, cur_group) in enumerate(z_gnn_corr_residual_groups):
    ax2.semilogx(sr.constants.PERIODS, z_gnn_res_lnVs30Diff_corr_std.loc[cur_key, sr.constants.PSA_KEYS], c=colors[i])

    if z_cim_corr_res_std is not None:
        ax2.semilogx(sr.constants.PERIODS, z_cim_res_lnVs30Diff_corr_std.loc[cur_key, sr.constants.PSA_KEYS], c=colors[i], linestyle="--")